# CSE4078 Spring 2026 - Deadline 1 Colab Notebook


In [ ]:
!pip -q install -U datasets transformers accelerate bitsandbytes huggingface_hub tqdm
!pip -q install -U sentence-transformers
!pip -q install --force-reinstall pandas==2.2.2 numpy==2.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 20.2 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from pathlib import Path
import json
import csv
import random
import re
import unicodedata
from collections import Counter, defaultdict

import pandas as pd
import torch
from tqdm.auto import tqdm

OUTPUT_ROOT = Path('/content/drive/MyDrive/CSE4078S26_GrpX_output')
DATA_DIR = OUTPUT_ROOT / 'data' / 'doktorsitesi'
BASELINE_DIR = OUTPUT_ROOT / 'baseline'
DATA_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = 'alibayram/doktorsitesi'
DATASET_SPLIT = 'train'
SEED = 42
TEST_SIZE = 1500

TARGET_MEDICAL_FIELDS = (
    'beyin-ve-sinir-cerrahisi',
    'uroloji',
    'ortopedi-ve-travmatoloji',
    'dahiliye-ve-ic-hastaliklari',
    'genel-cerrahi',
    'kulak-burun-bogaz-hastaliklari',
    'fiziksel-tip-ve-rehabilitasyon',
    'kardiyoloji',
    'kalp-damar-cerrahisi',
)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Output:', OUTPUT_ROOT)

CUDA: False
Output: /content/drive/MyDrive/CSE4078S26_GrpX_output


## Dataset Split Hazirlama

Fine-tuning sadece `sft_train.jsonl` ile yapilacak. `benchmark_test.jsonl` baseline ve final evaluation icin unseen kalacak.

In [ ]:
from datasets import load_dataset

def dedup_key(row):
    return (
        ' '.join(str(row.get('question_content', '')).split()).casefold(),
        ' '.join(str(row.get('question_answer', '')).split()).casefold(),
        str(row.get('doctor_speciality', '')).strip().casefold(),
    )

def format_sft_record(row):
    return {
        'instruction': 'A?a??daki t?bbi soruyu T?rk?e ve dikkatli bi?imde yan?tla.',
        'input': str(row['question_content']).strip(),
        'output': str(row['question_answer']).strip(),
        'doctor_speciality': str(row['doctor_speciality']).strip(),
    }

def proportional_take(by_label, total):
    import math
    available = {label: len(rows) for label, rows in by_label.items()}
    available_total = sum(available.values())
    raw_targets = {label: (count / available_total) * total for label, count in available.items()}
    targets = {label: min(available[label], math.floor(raw)) for label, raw in raw_targets.items()}
    remaining = total - sum(targets.values())
    labels = sorted(
        raw_targets,
        key=lambda label: (raw_targets[label] - math.floor(raw_targets[label]), available[label]),
        reverse=True,
    )
    while remaining:
        progressed = False
        for label in labels:
            if targets[label] < available[label]:
                targets[label] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise ValueError('Could not allocate requested stratified sample.')
    sampled = []
    for label, count in targets.items():
        sampled.extend(by_label[label][:count])
    return sampled

def create_project_train_test_split(rows, test_size=1500, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    seen = set()
    for row in rows:
        label = str(row.get('doctor_speciality', '')).strip()
        key = dedup_key(row)
        if label in TARGET_MEDICAL_FIELDS and key not in seen:
            seen.add(key)
            by_label[label].append(row)
    if not by_label:
        raise ValueError('No rows matched the required medical fields.')
    for bucket in by_label.values():
        rng.shuffle(bucket)
    total_available = sum(len(bucket) for bucket in by_label.values())
    if test_size >= total_available:
        raise ValueError('test_size leaves no training rows.')
    test = proportional_take(by_label, test_size)
    test_ids = {id(row) for row in test}
    train = [row for bucket in by_label.values() for row in bucket if id(row) not in test_ids]
    rng.shuffle(train)
    rng.shuffle(test)
    return train, test

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(format_sft_record(row), ensure_ascii=False))
            f.write("\n")


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).open(encoding='utf-8') if line.strip()]

In [ ]:
dataset = load_dataset(DATASET_ID, split=DATASET_SPLIT)
train_rows, test_rows = create_project_train_test_split(dataset, test_size=TEST_SIZE, seed=SEED)

write_jsonl(DATA_DIR / 'sft_train.jsonl', train_rows)
write_jsonl(DATA_DIR / 'benchmark_test.jsonl', test_rows)

summary = {
    'dataset': DATASET_ID,
    'train_size': len(train_rows),
    'benchmark_test_size': len(test_rows),
    'seed': SEED,
    'note': 'Fine-tuning uses only sft_train.jsonl. benchmark_test.jsonl remains unseen during training.',
}
(DATA_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
summary

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/66.5M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

(…)tle_spec_combined-00000-of-00001.parquet:   0%|          | 0.00/83.0M [00:00<?, ?B/s]

balanced_2_title-00000-of-00001.parquet:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/37527 [00:00<?, ? examples/s]

Generating title_spec_combined split:   0%|          | 0/187632 [00:00<?, ? examples/s]

Generating balanced_2_title split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating balanced_2_title_test split:   0%|          | 0/800 [00:00<?, ? examples/s]

{'dataset': 'alibayram/doktorsitesi',
 'train_size': 39612,
 'benchmark_test_size': 1500,
 'seed': 42,
 'note': 'Fine-tuning uses only sft_train.jsonl. benchmark_test.jsonl remains unseen during training.'}

In [ ]:
train = load_jsonl(DATA_DIR / 'sft_train.jsonl')
test = load_jsonl(DATA_DIR / 'benchmark_test.jsonl')
train_keys = {(r['input'], r['output'], r['doctor_speciality']) for r in train}
test_keys = {(r['input'], r['output'], r['doctor_speciality']) for r in test}

print('train rows:', len(train))
print('test rows:', len(test))
print('overlap:', len(train_keys & test_keys))
print('train distribution:', Counter(r['doctor_speciality'] for r in train))
print('test distribution:', Counter(r['doctor_speciality'] for r in test))

train rows: 39612
test rows: 1500
overlap: 0
train distribution: Counter({'beyin-ve-sinir-cerrahisi': 10150, 'uroloji': 8274, 'ortopedi-ve-travmatoloji': 5398, 'genel-cerrahi': 4799, 'kulak-burun-bogaz-hastaliklari': 3764, 'fiziksel-tip-ve-rehabilitasyon': 3559, 'kardiyoloji': 1897, 'kalp-damar-cerrahisi': 1017, 'dahiliye-ve-ic-hastaliklari': 754})
test distribution: Counter({'beyin-ve-sinir-cerrahisi': 384, 'uroloji': 313, 'ortopedi-ve-travmatoloji': 204, 'genel-cerrahi': 182, 'kulak-burun-bogaz-hastaliklari': 142, 'fiziksel-tip-ve-rehabilitasyon': 135, 'kardiyoloji': 72, 'kalp-damar-cerrahisi': 39, 'dahiliye-ve-ic-hastaliklari': 29})


## Baseline Evaluation Helpers

`USE_4BIT=True` Colab T4/L4 gibi sinirli VRAM ortamlarinda 4B modelleri sigdirmak icindir. A100 gibi guclu GPU varsa `False` yapabilirsin.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

USE_4BIT = True
MAX_NEW_TOKENS = 256

def normalize_text(text):
    text = unicodedata.normalize('NFKD', text.lower())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return ' '.join(text.split())

def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0

def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = sum((Counter(pred_tokens) & Counter(ref_tokens)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def load_model_and_tokenizer(model_id, trust_remote_code=False):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    kwargs = dict(device_map='auto', torch_dtype='auto', trust_remote_code=trust_remote_code)
    if USE_4BIT:
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    return model, tokenizer

def encode_prompt(tokenizer, question, device, enable_thinking=True):
    messages = [{'role': 'user', 'content': question.strip()}]
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = "Soru: " + question.strip() + chr(10) + "Cevap:"
    return tokenizer(text, return_tensors='pt').to(device)

@torch.inference_mode()
def generate_answer(model, tokenizer, question, max_new_tokens=MAX_NEW_TOKENS, enable_thinking=True):
    inputs = encode_prompt(tokenizer, question, model.device, enable_thinking=enable_thinking)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def evaluate_model(model_id, out_csv, trust_remote_code=False, limit=None, max_new_tokens=MAX_NEW_TOKENS, enable_thinking=True):
    test_rows = load_jsonl(DATA_DIR / 'benchmark_test.jsonl')
    if limit is not None:
        test_rows = test_rows[:limit]
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    done = set()
    if out_csv.exists():
        with out_csv.open(encoding='utf-8', newline='') as f:
            for row in csv.DictReader(f):
                done.add(int(row['index']))

    model, tokenizer = load_model_and_tokenizer(model_id, trust_remote_code=trust_remote_code)
    fieldnames = ['model', 'index', 'exact_match', 'token_f1', 'doctor_speciality', 'question', 'reference', 'prediction']
    write_header = not out_csv.exists()

    with out_csv.open('a', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        for idx, row in tqdm(list(enumerate(test_rows)), desc=model_id):
            if idx in done:
                continue
            pred = generate_answer(model, tokenizer, row['input'], max_new_tokens=max_new_tokens, enable_thinking=enable_thinking)
            writer.writerow({
                'model': model_id,
                'index': idx,
                'exact_match': exact_match(pred, row['output']),
                'token_f1': token_f1(pred, row['output']),
                'doctor_speciality': row['doctor_speciality'],
                'question': row['input'],
                'reference': row['output'],
                'prediction': pred,
            })
            f.flush()

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.read_csv(out_csv)

## Smoke Test

Once bu iki hucreyi calistir. Bunlar sadece 3 ornekle sistem kontroludur; odeve konacak sonuc degildir.

In [ ]:
qwen_debug = evaluate_model(
    'Qwen/Qwen3-4B',
    BASELINE_DIR / 'qwen3_4b_debug.csv',
    limit=3,
    max_new_tokens=64,
    enable_thinking=False,
)
qwen_debug[['exact_match', 'token_f1', 'prediction']].head()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
phi_debug = evaluate_model(
    "microsoft/Phi-3.5-mini-instruct",
    BASELINE_DIR / "phi_3_5_mini_debug.csv",
    trust_remote_code=False,
    limit=3,
    max_new_tokens=64,
    enable_thinking=False,
)
phi_debug[["exact_match", "token_f1", "prediction"]].head()


## Full Baseline Runs

Bu hucreler 1500 test orneginin tamamini calistirir. Runtime koparsa ayni hucreyi tekrar calistir; CSV checkpoint oldugu icin tamamlanan indexleri atlar.

In [ ]:
def evaluate_model_on_file(
    model_id,
    test_file,
    out_csv,
    trust_remote_code=False,
    max_new_tokens=96,
    enable_thinking=False,
):
    test_rows = load_jsonl(test_file)

    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    done = set()
    if out_csv.exists():
        with out_csv.open(encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                done.add(int(row["index"]))

    model, tokenizer = load_model_and_tokenizer(
        model_id,
        trust_remote_code=trust_remote_code,
    )

    fieldnames = [
        "model",
        "index",
        "exact_match",
        "token_f1",
        "doctor_speciality",
        "question",
        "reference",
        "prediction",
    ]

    write_header = not out_csv.exists()

    with out_csv.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()

        for idx, row in tqdm(list(enumerate(test_rows)), desc=model_id):
            if idx in done:
                continue

            pred = generate_answer(
                model,
                tokenizer,
                row["input"],
                max_new_tokens=max_new_tokens,
                enable_thinking=enable_thinking,
            )

            writer.writerow({
                "model": model_id,
                "index": idx,
                "exact_match": exact_match(pred, row["output"]),
                "token_f1": token_f1(pred, row["output"]),
                "doctor_speciality": row["doctor_speciality"],
                "question": row["input"],
                "reference": row["output"],
                "prediction": pred,
            })
            f.flush()

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.read_csv(out_csv)


In [ ]:
def make_stratified_eval_sample(in_path, out_path, sample_size=1500, seed=42):
    rows = load_jsonl(in_path)
    rng = random.Random(seed)

    by_label = defaultdict(list)
    for row in rows:
        by_label[row["doctor_speciality"]].append(row)

    total = len(rows)
    raw_targets = {
        label: len(items) / total * sample_size
        for label, items in by_label.items()
    }

    targets = {
        label: int(raw_targets[label])
        for label in by_label
    }

    remaining = sample_size - sum(targets.values())
    labels = sorted(
        by_label,
        key=lambda label: raw_targets[label] - targets[label],
        reverse=True,
    )

    for label in labels[:remaining]:
        targets[label] += 1

    sample = []
    for label, items in by_label.items():
        rng.shuffle(items)
        sample.extend(items[:targets[label]])

    rng.shuffle(sample)

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with out_path.open("w", encoding="utf-8") as f:
        for row in sample:
            f.write(json.dumps(row, ensure_ascii=False))
            f.write("\n")

    return pd.DataFrame(sample)["doctor_speciality"].value_counts()


In [ ]:
make_stratified_eval_sample(
    DATA_DIR / "benchmark_test.jsonl",
    DATA_DIR / "benchmark_test_1500.jsonl",
    sample_size=1500,
    seed=42,
)


,count
doctor_speciality,
beyin-ve-sinir-cerrahisi,384
uroloji,313
ortopedi-ve-travmatoloji,204
genel-cerrahi,182
kulak-burun-bogaz-hastaliklari,142
fiziksel-tip-ve-rehabilitasyon,135
kardiyoloji,72
kalp-damar-cerrahisi,39
dahiliye-ve-ic-hastaliklari,29


In [ ]:
from pathlib import Path

Path(BASELINE_DIR / "qwen3_4b_baseline_1500.csv").unlink(missing_ok=True)

qwen_1500 = evaluate_model_on_file(
    "Qwen/Qwen3-4B",
    DATA_DIR / "benchmark_test_1500.jsonl",
    BASELINE_DIR / "qwen3_4b_baseline_1500.csv",
    max_new_tokens=96,
    enable_thinking=False,
)

qwen_1500.tail()


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen/Qwen3-4B:   0%|          | 0/1500 [00:00<?, ?it/s]

,model,index,exact_match,token_f1,doctor_speciality,question,reference,prediction
1495,Qwen/Qwen3-4B,1495,0.0,0.000000,genel-cerrahi,MERHABALAR ALAATTİN BEY BENİM EŞİMİN 3-4 GÜN Ö...,Bence endişe edecek bir durum değil. Derialtı ...,"Merhaba, Alaattin Bey. İlk olarak, size bu kon..."
1496,Qwen/Qwen3-4B,1496,0.0,0.065217,beyin-ve-sinir-cerrahisi,meraba. bel fıtığım var. sabahları uyandığım z...,Tedavinize karar vermek için sizi görmemde yar...,"Merhaba! Bel fıtığı (spondilozis, discal herni..."
1497,Qwen/Qwen3-4B,1497,0.0,0.042553,dahiliye-ve-ic-hastaliklari,HOCAM MERHABA BEN TUĞBA EFENDİM 26 YAŞINDAYIM ...,Kanda iltihaplanma olması çok kötü birşey deği...,"Merhaba Tuğba Efendim, merak etmenin nedeni, k..."
1498,Qwen/Qwen3-4B,1498,0.0,0.000000,kulak-burun-bogaz-hastaliklari,hocam benim küçüklüğümden beri sol kulak zarım...,kulak zarı delikse mutlaka ameliyat olmalısını...,"Merhaba, çok detaylı bir sorunuz var. İlk olar..."
1499,Qwen/Qwen3-4B,1499,0.0,0.136364,kulak-burun-bogaz-hastaliklari,hocam merhabalar benim sorum kızımda geniz eti...,Geniz eti olup olmadığı KBB muayenesi ile kola...,"Merhaba, sorunuz oldukça önemli ve detaylı. Ge..."


In [ ]:
from pathlib import Path

Path(BASELINE_DIR / "phi_3_5_mini_baseline_1500.csv").unlink(missing_ok=True)

phi_300 = evaluate_model_on_file(
    "microsoft/Phi-3.5-mini-instruct",
    DATA_DIR / "benchmark_test_1500.jsonl",
    BASELINE_DIR / "phi_3_5_mini_baseline_1500.csv",
    trust_remote_code=False,
    max_new_tokens=96,
    enable_thinking=False,
)

phi_1500.tail()


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

microsoft/Phi-3.5-mini-instruct:   0%|          | 0/1500 [00:00<?, ?it/s]

NameError: name 'phi_1500' is not defined

## Sonuclari Ozetle

In [ ]:
def summarize_result(path):
    df = pd.read_csv(path)
    return {
        "file": str(path),
        "rows": len(df),
        "exact_match": float(df["exact_match"].mean()),
        "token_f1": float(df["token_f1"].mean()),
    }

summary_rows = []
for file in [
    BASELINE_DIR / "qwen3_4b_baseline_1500.csv",
    BASELINE_DIR / "phi_3_5_mini_baseline_1500.csv",
]:
    if file.exists():
        summary_rows.append(summarize_result(file))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(BASELINE_DIR / "baseline_summary_1500.csv", index=False)
summary_df


,file,rows,exact_match,token_f1
0,/content/drive/MyDrive/CSE4078S26_GrpX_output/...,1500,0.0,0.065312
1,/content/drive/MyDrive/CSE4078S26_GrpX_output/...,1500,0.0,0.054228


In [ ]:
for file in [
    BASELINE_DIR / "qwen3_4b_baseline_1500.csv",
    BASELINE_DIR / "phi_3_5_mini_baseline_1500.csv",
]:
    if file.exists():
        df = pd.read_csv(file)
        by_field = df.groupby("doctor_speciality", as_index=False).agg(
            rows=("index", "count"),
            token_f1=("token_f1", "mean"),
            exact_match=("exact_match", "mean"),
        )

        out = BASELINE_DIR / (Path(file).stem + "_by_field.csv")
        by_field.to_csv(out, index=False)

        print(out)
        display(by_field.sort_values("token_f1", ascending=False))


/content/drive/MyDrive/CSE4078S26_GrpX_output/baseline/qwen3_4b_baseline_1500_by_field.csv


,doctor_speciality,rows,token_f1,exact_match
2,fiziksel-tip-ve-rehabilitasyon,135,0.085205,0.0
5,kardiyoloji,72,0.084647,0.0
7,ortopedi-ve-travmatoloji,204,0.070056,0.0
3,genel-cerrahi,182,0.069381,0.0
1,dahiliye-ve-ic-hastaliklari,29,0.064045,0.0
6,kulak-burun-bogaz-hastaliklari,142,0.063242,0.0
8,uroloji,313,0.059025,0.0
4,kalp-damar-cerrahisi,39,0.057276,0.0
0,beyin-ve-sinir-cerrahisi,384,0.057045,0.0


/content/drive/MyDrive/CSE4078S26_GrpX_output/baseline/phi_3_5_mini_baseline_1500_by_field.csv


,doctor_speciality,rows,token_f1,exact_match
5,kardiyoloji,72,0.067207,0.0
2,fiziksel-tip-ve-rehabilitasyon,135,0.066984,0.0
1,dahiliye-ve-ic-hastaliklari,29,0.059696,0.0
7,ortopedi-ve-travmatoloji,204,0.054743,0.0
3,genel-cerrahi,182,0.053671,0.0
6,kulak-burun-bogaz-hastaliklari,142,0.053316,0.0
8,uroloji,313,0.051979,0.0
0,beyin-ve-sinir-cerrahisi,384,0.049677,0.0
4,kalp-damar-cerrahisi,39,0.048119,0.0


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cpu"
)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def add_semantic_similarity(csv_path):
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    references = df["reference"].fillna("").astype(str).tolist()
    predictions = df["prediction"].fillna("").astype(str).tolist()

    ref_emb = embedder.encode(
        references,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    pred_emb = embedder.encode(
        predictions,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    similarities = (ref_emb * pred_emb).sum(axis=1)

    df["semantic_similarity"] = similarities

    out_path = csv_path.with_name(csv_path.stem + "_semantic.csv")
    df.to_csv(out_path, index=False)

    return df, out_path


In [ ]:
qwen_sem, qwen_sem_path = add_semantic_similarity(
    BASELINE_DIR / "qwen3_4b_baseline_1500.csv"
)

phi_sem, phi_sem_path = add_semantic_similarity(
    BASELINE_DIR / "phi_3_5_mini_baseline_1500.csv"
)

print(qwen_sem_path)
print(phi_sem_path)


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

/content/drive/MyDrive/CSE4078S26_GrpX_output/baseline/qwen3_4b_baseline_1500_semantic.csv
/content/drive/MyDrive/CSE4078S26_GrpX_output/baseline/phi_3_5_mini_baseline_1500_semantic.csv


In [ ]:
semantic_summary = pd.DataFrame([
    {
        "model": "Qwen/Qwen3-4B",
        "rows": len(qwen_sem),
        "exact_match": qwen_sem["exact_match"].mean(),
        "token_f1": qwen_sem["token_f1"].mean(),
        "semantic_similarity": qwen_sem["semantic_similarity"].mean(),
    },
    {
        "model": "microsoft/Phi-3.5-mini-instruct",
        "rows": len(phi_sem),
        "exact_match": phi_sem["exact_match"].mean(),
        "token_f1": phi_sem["token_f1"].mean(),
        "semantic_similarity": phi_sem["semantic_similarity"].mean(),
    },
])

semantic_summary.to_csv(
    BASELINE_DIR / "baseline_summary_1500_semantic.csv",
    index=False
)

semantic_summary


,model,rows,exact_match,token_f1,semantic_similarity
0,Qwen/Qwen3-4B,1500,0.0,0.065312,0.379796
1,microsoft/Phi-3.5-mini-instruct,1500,0.0,0.054228,0.342425


In [ ]:
import pandas as pd
from pathlib import Path

def add_length_stats(csv_path, output_path=None):
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    preds = df["prediction"].fillna("").astype(str)

    df["generated_char_length"] = preds.str.len()
    df["generated_token_length"] = preds.apply(lambda x: len(x.split()))

    summary = {
        "file": csv_path.name,
        "rows": len(df),
        "avg_generated_chars": df["generated_char_length"].mean(),
        "median_generated_chars": df["generated_char_length"].median(),
        "avg_generated_tokens": df["generated_token_length"].mean(),
        "median_generated_tokens": df["generated_token_length"].median(),
        "min_generated_tokens": df["generated_token_length"].min(),
        "max_generated_tokens": df["generated_token_length"].max(),
    }

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)

    return df, summary


qwen_len_df, qwen_len_summary = add_length_stats(
    BASELINE_DIR / "qwen3_4b_baseline_1500_semantic.csv",
    BASELINE_DIR / "qwen3_4b_baseline_1500_semantic_length.csv",
)

phi_len_df, phi_len_summary = add_length_stats(
    BASELINE_DIR / "phi_3_5_mini_baseline_1500_semantic.csv",
    BASELINE_DIR / "phi_3_5_mini_baseline_1500_semantic_length.csv",
)

length_summary_1500 = pd.DataFrame([
    qwen_len_summary,
    phi_len_summary,
])

length_summary_1500.to_csv(
    BASELINE_DIR / "baseline_length_summary_1500.csv",
    index=False
)

length_summary_1500


,file,rows,avg_generated_chars,median_generated_chars,avg_generated_tokens,median_generated_tokens,min_generated_tokens,max_generated_tokens
0,qwen3_4b_baseline_1500_semantic.csv,1500,271.973333,271.0,36.417333,36.0,25,75
1,phi_3_5_mini_baseline_1500_semantic.csv,1500,198.379333,198.0,26.600000,27.0,6,67


In [ ]:
!pip -q install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from pathlib import Path
from bert_score import score
import torch

def add_bertscore(csv_path, output_path=None, model_type="xlm-roberta-base", batch_size=16):
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    predictions = df["prediction"].fillna("").astype(str).tolist()
    references = df["reference"].fillna("").astype(str).tolist()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running BERTScore on {csv_path.name}")
    print(f"Rows: {len(df)} | Device: {device} | Model: {model_type}")

    P, R, F1 = score(
        predictions,
        references,
        model_type=model_type,
        lang="tr",
        batch_size=batch_size,
        device=device,
        verbose=True,
        rescale_with_baseline=False,
    )

    df["bertscore_precision"] = P.cpu().numpy()
    df["bertscore_recall"] = R.cpu().numpy()
    df["bertscore_f1"] = F1.cpu().numpy()

    summary = {
        "file": csv_path.name,
        "rows": len(df),
        "bertscore_precision": df["bertscore_precision"].mean(),
        "bertscore_recall": df["bertscore_recall"].mean(),
        "bertscore_f1": df["bertscore_f1"].mean(),
    }

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)

    return df, summary


qwen_bert_df, qwen_bert_summary = add_bertscore(
    BASELINE_DIR / "qwen3_4b_baseline_1500_semantic.csv",
    BASELINE_DIR / "qwen3_4b_baseline_1500_semantic_bertscore.csv",
    model_type="xlm-roberta-base",
    batch_size=16,
)

phi_bert_df, phi_bert_summary = add_bertscore(
    BASELINE_DIR / "phi_3_5_mini_baseline_1500_semantic.csv",
    BASELINE_DIR / "phi_3_5_mini_baseline_1500_semantic_bertscore.csv",
    model_type="xlm-roberta-base",
    batch_size=16,
)

bertscore_summary_1500 = pd.DataFrame([
    qwen_bert_summary,
    phi_bert_summary,
])

bertscore_summary_1500.to_csv(
    BASELINE_DIR / "baseline_bertscore_summary_1500.csv",
    index=False
)

bertscore_summary_1500


Running BERTScore on qwen3_4b_baseline_1500_semantic.csv
Rows: 1500 | Device: cpu | Model: xlm-roberta-base


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/184 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/94 [00:00<?, ?it/s]

done in 505.79 seconds, 2.97 sentences/sec


,file,rows,bertscore_precision,bertscore_recall,bertscore_f1
0,qwen3_4b_baseline_1500_semantic.csv,1500,0.811160,0.827803,0.819128
1,phi_3_5_mini_baseline_1500_semantic.csv,1500,0.814854,0.822183,0.818255


In [ ]:
!pip -q install rouge-score

import pandas as pd
from pathlib import Path
from rouge_score import rouge_scorer

def add_rouge_scores(csv_path, output_path=None):
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    predictions = df["prediction"].fillna("").astype(str)
    references = df["reference"].fillna("").astype(str)

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=False
    )

    rouge1_precision = []
    rouge1_recall = []
    rouge1_f1 = []

    rouge2_precision = []
    rouge2_recall = []
    rouge2_f1 = []

    rougeL_precision = []
    rougeL_recall = []
    rougeL_f1 = []

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)

        rouge1_precision.append(scores["rouge1"].precision)
        rouge1_recall.append(scores["rouge1"].recall)
        rouge1_f1.append(scores["rouge1"].fmeasure)

        rouge2_precision.append(scores["rouge2"].precision)
        rouge2_recall.append(scores["rouge2"].recall)
        rouge2_f1.append(scores["rouge2"].fmeasure)

        rougeL_precision.append(scores["rougeL"].precision)
        rougeL_recall.append(scores["rougeL"].recall)
        rougeL_f1.append(scores["rougeL"].fmeasure)

    df["rouge1_precision"] = rouge1_precision
    df["rouge1_recall"] = rouge1_recall
    df["rouge1_f1"] = rouge1_f1

    df["rouge2_precision"] = rouge2_precision
    df["rouge2_recall"] = rouge2_recall
    df["rouge2_f1"] = rouge2_f1

    df["rougeL_precision"] = rougeL_precision
    df["rougeL_recall"] = rougeL_recall
    df["rougeL_f1"] = rougeL_f1

    summary = {
        "file": csv_path.name,
        "rows": len(df),
        "rouge1_precision": df["rouge1_precision"].mean(),
        "rouge1_recall": df["rouge1_recall"].mean(),
        "rouge1_f1": df["rouge1_f1"].mean(),
        "rouge2_precision": df["rouge2_precision"].mean(),
        "rouge2_recall": df["rouge2_recall"].mean(),
        "rouge2_f1": df["rouge2_f1"].mean(),
        "rougeL_precision": df["rougeL_precision"].mean(),
        "rougeL_recall": df["rougeL_recall"].mean(),
        "rougeL_f1": df["rougeL_f1"].mean(),
    }

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)

    return df, summary


qwen_rouge_df, qwen_rouge_summary = add_rouge_scores(
    BASELINE_DIR / "qwen3_4b_baseline_1500_semantic.csv",
    BASELINE_DIR / "qwen3_4b_baseline_1500_semantic_rouge.csv",
)

phi_rouge_df, phi_rouge_summary = add_rouge_scores(
    BASELINE_DIR / "phi_3_5_mini_baseline_1500_semantic.csv",
    BASELINE_DIR / "phi_3_5_mini_baseline_1500_semantic_rouge.csv",
)

rouge_summary_1500 = pd.DataFrame([
    qwen_rouge_summary,
    phi_rouge_summary,
])

rouge_summary_1500.to_csv(
    BASELINE_DIR / "baseline_rouge_summary_1500.csv",
    index=False
)

rouge_summary_1500


,file,rows,rouge1_precision,rouge1_recall,rouge1_f1,rouge2_precision,rouge2_recall,rouge2_f1,rougeL_precision,rougeL_recall,rougeL_f1
0,qwen3_4b_baseline_1500_semantic.csv,1500,0.108932,0.155859,0.110888,0.013837,0.019159,0.013966,0.071632,0.113077,0.075719
1,phi_3_5_mini_baseline_1500_semantic.csv,1500,0.115775,0.135267,0.106862,0.014584,0.016696,0.013360,0.080232,0.102533,0.076907


## Bana Gonderecegin Dosyalar

Drive'da su klasore bak:

`MyDrive/CSE4078S26_GrpX_output/baseline/`

Bana ozellikle sunlari gonder:

- `baseline_summary.csv`
- `qwen3_4b_baseline.csv`
- `phi_3_5_mini_baseline.csv`
- varsa `*_by_field.csv` dosyalari